# Olist SQL — executed verification notebook
Actual Olist CSVs; guide v2; purchase window [2017-01-01, 2018-08-01).
This notebook was executed by `the original standard-library executor (retained in the source package)` using Python exec in a fresh process. nbclient and nbformat packages were unavailable. It creates a temporary database, runs actual queries, and compares their outputs with delivered results. Synthetic tests run separately and are explicitly labeled. No synthetic row enters real-data results.

**Historical execution evidence.** The seven code cells and their execution counts/outputs originate from the supplied executed notebook. Source paths were adapted after that execution for this repository, and one temporary-directory string in stdout was replaced with a placeholder. The edited source has not been executed as a notebook; use the independently tested CLI pipeline in [README](../README.md) to reproduce. The original Python/SQLite/library version output describes the historical run.

Public copy provenance: [notebook transformation log](../evidence/notebook_transformations.json). Input CSVs must be placed in `data/raw/` to rerun these cells. Synthetic test output is labeled separately.


In [1]:
import sys,sqlite3,json,subprocess,tempfile,hashlib
from pathlib import Path
import pandas as pd
root=Path.cwd()
if root.name == "notebooks": root=root.parent
if not (root/"sql/analysis.sql").is_file(): raise RuntimeError("Start in repository root or notebooks directory.")
def run(*args):
    p=subprocess.run([sys.executable,*args],text=True,capture_output=True,cwd=root)
    print(p.stdout,end='');print(p.stderr,end='')
    if p.returncode: raise RuntimeError(f"Command failed: {args}; exit={p.returncode}")
print(json.dumps({'python':sys.version,'sqlite':sqlite3.sqlite_version,'pandas':pd.__version__},indent=2))


{
  "python": "3.12.13 (main, Aug  7 2026, 02:25:39) [Clang 22.1.3 ]",
  "sqlite": "3.53.1",
  "pandas": "2.2.3"
}


In [2]:
meta=json.loads((root/'evidence/source_metadata.json').read_text())
manifest=json.loads((root/'evidence/input_manifest.json').read_text())
for item in manifest.values():
    assert hashlib.sha256((root/'data/raw'/item['file']).read_bytes()).hexdigest()==item['sha256']
print('All nine original CSV SHA-256 hashes match recorded inputs.')
print('Source version field:',meta['currentVersionNumber'],'updated:',meta['lastUpdated'],'license:',meta['licenseName'])
print(pd.DataFrame(manifest).T[['file','rows']].to_string())


All nine original CSV SHA-256 hashes match recorded inputs.
Source version field: 2 updated: 2021-10-01T19:08:27.97Z license: CC BY-NC-SA 4.0
                                                       file     rows
customers                       olist_customers_dataset.csv    99441
geolocation                   olist_geolocation_dataset.csv  1000163
order_items                   olist_order_items_dataset.csv   112650
payments                   olist_order_payments_dataset.csv   103886
reviews                     olist_order_reviews_dataset.csv    99224
orders                             olist_orders_dataset.csv    99441
products                         olist_products_dataset.csv    32951
sellers                           olist_sellers_dataset.csv     3095
category_translation  product_category_name_translation.csv       71


In [3]:
temp=tempfile.TemporaryDirectory(prefix='olist_clean_');work=Path(temp.name)
run('src/audit_raw.py','--data','data/raw','--out',str(work/'audit'))
run('src/import_olist.py','--data','data/raw','--db',str(work/'clean.db'),'--start','2017-01-01','--end','2018-08-01')
run('src/run_analysis.py','--db',str(work/'clean.db'),'--out',str(work/'results'))
for i in range(1,13):
    pd.testing.assert_frame_equal(pd.read_csv(work/'results'/f'Q{i:02}.csv'),pd.read_csv(root/'results'/f'Q{i:02}.csv'))
print('Clean re-import: all 12 SQL result tables match delivered tables.')


{
  "rows": {
    "customers": 99441,
    "geolocation": 1000163,
    "order_items": 112650,
    "payments": 103886,
    "reviews": 99224,
    "orders": 99441,
    "products": 32951,
    "sellers": 3095,
    "category_translation": 71
  },
  "audit_directory": "<temporary-directory>/audit"
}
{
  "QA": "PASS",
  "n_checks": 20,
  "metrics": {
    "eligible_orders": 89860,
    "orders_with_items": 89860,
    "orders_without_items": 0,
    "n_items": 102738,
    "item_sales_brl": 12342450.49,
    "n_category_buckets": 74,
    "n_reviewed_eligible_orders": 89238,
    "review_coverage_pct": 99.3078121522368,
    "n_buyers_with_valid_id": 86960,
    "eligible_orders_missing_buyer_id": 0,
    "multi_category_orders": 708,
    "unknown_seller_items": 0,
    "delivery": {
      "eligible_orders": 89860,
      "missing_invalid_dates": 8,
      "delivered_before_purchase": 0,
      "same_date_orders": 1024
    },
    "window": {
      "start_date": "2017-01-01",
      "end_date": "2018-08-01",
  

In [4]:
for i in range(1,13):
    d=pd.read_csv(work/'results'/f'Q{i:02}.csv')
    print(f'\nQ{i:02}: {len(d)} output rows; first 15 shown (full table in results).')
    print(d.head(15).to_string(index=False))



Q01: 8 output rows; first 15 shown (full table in results).
order_status  n_orders
   delivered     96478
     shipped      1107
    canceled       625
 unavailable       609
    invoiced       314
  processing       301
     created         5
    approved         2

Q02: 8 output rows; first 15 shown (full table in results).
order_status      first_purchase       last_purchase  n_orders
    approved 2017-02-06 20:18:17 2017-04-25 01:25:34         2
    canceled 2016-09-05 00:15:34 2018-10-17 17:30:18       625
     created 2017-11-06 13:12:34 2018-02-09 17:21:04         5
   delivered 2016-09-15 12:16:38 2018-08-29 15:00:37     96478
    invoiced 2016-10-04 13:02:10 2018-08-14 18:45:08       314
  processing 2016-10-05 22:44:13 2018-07-23 18:03:03       301
     shipped 2016-09-04 21:15:19 2018-09-03 09:06:57      1107
 unavailable 2016-10-05 14:16:28 2018-08-21 12:21:00       609

Q03: 15 output rows; first 15 shown (full table in results).
             category  n_orders  n_items  

In [5]:
print(pd.read_csv(work/'evidence/qa_checks.csv').to_string(index=False))
print('\nWindow sensitivity (actual data):')
print(pd.read_csv(work/'results/window_sensitivity.csv').to_string(index=False))
print('\nReview-policy sensitivity (actual data; ratings rounded to 2 decimals):')
print(pd.read_csv(work/'results/review_policy_sensitivity.csv').to_string(index=False))


                       check  passed     actual   expected
    raw_to_item_base_n_items    True     102738     102738
   raw_to_item_base_n_orders    True      89860      89860
raw_to_item_base_sales_cents    True 1234245049 1234245049
        category_sales_cents    True 1234245049 1234245049
           review_one_unique    True          0          0
   Q11_valid_delivery_orders    True      89852      89852
 same_calendar_date_not_late    True       1024       1024
               Q07_monotonic    True       True       True
                Q07_ends_100    True      100.0        100
               Q12_monotonic    True       True       True
                Q12_ends_100    True      100.0        100
         Q04_sales_reconcile    True 1234245049 1234245049
         Q05_sales_reconcile    True 1234245049 1234245049
         Q07_sales_reconcile    True 1234245049 1234245049
         Q09_sales_reconcile    True 1234245049 1234245049
         Q12_sales_reconcile    True 1234245049 12342450

In [6]:
print('SYNTHETIC REGRESSION TESTS — not Olist business observations')
run('tests/test_olist.py')


SYNTHETIC REGRESSION TESTS — not Olist business observations
test_all_12_execute (__main__.TestOlist.test_all_12_execute) ... ok
test_calendar_same_date_not_late (__main__.TestOlist.test_calendar_same_date_not_late) ... ok
test_import_rejects_blank_and_normalized_item_keys (__main__.TestOlist.test_import_rejects_blank_and_normalized_item_keys) ... ok
test_import_rejects_duplicate_keys_and_nonfinite_prices (__main__.TestOlist.test_import_rejects_duplicate_keys_and_nonfinite_prices) ... ok
test_import_roundtrip_and_overwrite_guard (__main__.TestOlist.test_import_roundtrip_and_overwrite_guard) ... ok
test_month_spine_and_year_separation (__main__.TestOlist.test_month_spine_and_year_separation) ... ok
test_pareto_and_full_sellers (__main__.TestOlist.test_pareto_and_full_sellers) ... ok
test_repeat_uses_unique_customer (__main__.TestOlist.test_repeat_uses_unique_customer) ... ok
test_review_dedup_and_valid_integer_rating (__main__.TestOlist.test_review_dedup_and_valid_integer_rating) ... ok

In [7]:
run('src/make_charts.py','--results',str(work/'results'),'--out',str(work/'figures'))
print('Saved figures from reconciled real-data result tables.')
temp.cleanup()


{
  "charts": [
    "01_category_pareto.png",
    "02_volume_price.png",
    "03_delivery_reviews.png",
    "04_monthly_top5.png",
    "05_freight_ratio.png",
    "06_seller_concentration.png"
  ],
  "matplotlib": "3.10.8"
}
Saved figures from reconciled real-data result tables.
